# Notebook 03 – kNN Collaborative Filtering

- Item-item kNN with Pearson similarity
- Sensitivity sweep: vary k
- Rating metrics (RMSE / MAE)
- Top-K ranking metrics (Precision, Recall, NDCG)
- Qualitative recommendations


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.knn_cf     import SurpriseKNN, knn_sensitivity, get_topk_recs_knn
from src.evaluation import rating_metrics, topk_metrics

sns.set_style('whitegrid')
os.makedirs('../results', exist_ok=True)
SEED = 42
np.random.seed(SEED)

In [ ]:
train  = pd.read_parquet('../data/train.parquet')
test   = pd.read_parquet('../data/test.parquet')
movies = pd.read_csv('../data/movie.csv')
print(f'Train: {len(train):,}   Test: {len(test):,}')

## 3.1 Sensitivity sweep: vary k

Trains on full training data for each k (kNN needs density for similarity).
Evaluates on 30% test subsample for speed.

In [ ]:
# NOTE: This cell is slow on full data (~5-6 hours for 5 k values)
# Results from run_all.py are already saved in results/knn_sensitivity.csv
# You can load those instead:
# df_sens = pd.read_csv('../results/knn_sensitivity.csv', index_col='k')

df_sens = knn_sensitivity(
    train, test,
    k_values=[10, 20, 40, 60, 80],
    seed=SEED
)
print(df_sens)

In [ ]:
# Load from saved CSV (fast — use this if you already ran run_all.py)
df_sens = pd.read_csv('../results/knn_sensitivity.csv', index_col='k')
print(df_sens)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_sens['RMSE'].plot(ax=axes[0], marker='o', color='steelblue')
axes[0].set_title('kNN – k vs RMSE')
axes[0].set_xlabel('k (number of neighbours)')
axes[0].set_ylabel('RMSE')

df_sens['MAE'].plot(ax=axes[1], marker='s', color='coral')
axes[1].set_title('kNN – k vs MAE')
axes[1].set_xlabel('k (number of neighbours)')
axes[1].set_ylabel('MAE')

plt.tight_layout()
plt.savefig('../results/knn_k_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.2 Train best kNN model (k=40)

In [ ]:
best_k = int(df_sens['RMSE'].idxmin())
print(f'Best k = {best_k}')

knn = SurpriseKNN(k=best_k, sim_name='pearson', user_based=False, min_support=3)
knn.fit(train)
knn_metrics = knn.evaluate(test, label=f'KNN k={best_k}')
print(knn_metrics)

## 3.3 Top-K Ranking Metrics

In [ ]:
sample_users = test['user_idx'].unique()[:500]
test_sample  = test[test['user_idx'].isin(sample_users)].copy()

test_sample['pred_rating'] = knn.predict_df(test_sample)
test_sample = test_sample.rename(columns={'rating': 'true_rating'})

print(f'Top-10 metrics for kNN (k={best_k}):')
topk_knn = topk_metrics(test_sample, k=10, threshold=4.0)
print(topk_knn)

## 3.4 Qualitative Recommendations

In [ ]:
sample_uid      = train['str_userId'].iloc[0]
rated_str       = set(train[train['str_userId'] == sample_uid]['str_movieId'])
all_str_mids    = train['str_movieId'].unique()

top10 = get_topk_recs_knn(knn.algo, sample_uid, all_str_mids, rated_str, k=10)

print(f'Top-10 recommendations for user {sample_uid} '
      f'(rated {len(rated_str)} movies):\n')
for rank, (mid, score) in enumerate(top10, 1):
    t     = movies[movies['movieId'] == int(mid)]['title'].values
    title = t[0] if len(t) > 0 else mid
    print(f'  {rank:>2}. {title:<55} predicted={score:.2f}')

In [ ]:
pd.DataFrame([knn_metrics]).to_csv('../results/knn_results.csv', index=False)
pd.DataFrame([topk_knn]).to_csv('../results/knn_topk_metrics.csv', index=False)
print('Saved.')